In [1]:
import os
base_path = os.getcwd()

In [2]:
import polars as pl

local_parquet_path = os.path.join(base_path, "data", "train.parquet")

# Menggunakan scan_parquet (Lazy) agar RAM tetap hemat
# Data hanya diproses saat .collect() dipanggil
df_pq = pl.scan_parquet(local_parquet_path)

In [3]:
df_sample = df_pq.head(15_000_000).collect()

print(f"Data berhasil disimpan di variabel 'df_sample'. Jumlah baris: {len(df_sample)}")
display(df_sample)

Data berhasil disimpan di variabel 'df_sample'. Jumlah baris: 15000000


ip,app,device,os,channel,click_time,attributed_time,is_attributed
i64,i64,i64,i64,i64,str,str,i64
83230,3,1,13,379,"""2017-11-06 14:32:21""",null,0
17357,3,1,19,379,"""2017-11-06 14:33:34""",null,0
35810,3,1,13,379,"""2017-11-06 14:34:12""",null,0
45745,14,1,13,478,"""2017-11-06 14:34:52""",null,0
161007,3,1,13,379,"""2017-11-06 14:35:08""",null,0
…,…,…,…,…,…,…,…
85065,12,2,40,265,"""2017-11-07 01:37:13""",null,0
113719,3,1,31,424,"""2017-11-07 01:37:13""",null,0
43547,3,1,13,480,"""2017-11-07 01:37:13""",null,0


In [4]:
from feature_engineering import generate_fraud_features, apply_heuristic_rules
from blocking_strategy import apply_business_blocking_strategy
df_sample = generate_fraud_features(df_sample)
df_sample = apply_heuristic_rules(df_sample)
df_sample = apply_business_blocking_strategy(df_sample)

In [5]:
import json
import xgboost as xgb

In [7]:
model_filename = os.path.join("model", "fraud_detection_model_v1.ubj")
schema_filename = os.path.join("model", "model_features_schema_v1.json")

In [8]:
with open(schema_filename, "r") as f:
    expected_features = json.load(f)

In [9]:
production_model = xgb.Booster()
production_model.load_model(model_filename)

In [10]:
expected_features_set = set(expected_features)

In [11]:
expected_features_set

{'ip_rhythm_std_1h',
 'ip_unique_apps_per_hour',
 'ip_unique_os_per_hour',
 'is_first_click'}

In [14]:
feature_data = df_sample.select(expected_features).to_numpy()
dmat = xgb.DMatrix(feature_data, feature_names=expected_features)
predictions = production_model.predict(dmat)

df_sample = df_sample.with_columns(
    pl.Series("bot_probability", predictions)
)

print("Predictions completed. Bot probability column added.")
display(df_sample.select(['bot_probability']).head())


Predictions completed. Bot probability column added.


bot_probability
f32
0.000231
0.000231
0.000231
0.000231
0.000231


In [16]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []

for thresh in thresholds:
    bot_count = df_sample.filter(pl.col('bot_probability') >= thresh).shape[0]
    human_count = df_sample.filter(pl.col('bot_probability') < thresh).shape[0]
    results.append({'threshold': thresh, 'bot_count': bot_count, 'human_count': human_count})

result_df = pl.DataFrame(results)
result_df.write_csv('aggregation_results.csv')
print("CSV file 'aggregation_results.csv' created successfully.")

CSV file 'aggregation_results.csv' created successfully.


In [17]:
result_df

threshold,bot_count,human_count
f64,i64,i64
0.1,14709432,290568
0.2,14574304,425696
0.3,13643546,1356454
0.4,11928112,3071888
0.5,10950962,4049038
0.6,10560644,4439356
0.7,10092536,4907464
0.8,9752599,5247401
0.9,9206215,5793785
